# 02 Single Factor Analysis

Run one candidate factor through IC diagnostics and layered return analysis, then export the core statistics for review.

In [ ]:
from pathlib import Path
import sys

import pandas as pd


def locate_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "apps").exists():
            return candidate
    raise RuntimeError("repo root not found")


REPO_ROOT = locate_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from apps.quant_platform.research.data_loader import ResearchDataLoader
from apps.quant_platform.research.scripts.run_single_factor import run_single_factor_analysis

RESEARCH_ROOT = REPO_ROOT / "apps/quant_platform/research"
OUTPUT_ROOT = RESEARCH_ROOT / "output/notebook_exports"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
loader = ResearchDataLoader()

In [ ]:
factor_col = "pct_chg"
panel = loader.prepare_panel(
    loader.load_panel(
        start_date="2024-01-01",
        end_date="2024-06-30",
        columns=["ts_code", "trade_date", "open", "close", "pct_chg", "turnover_rate_f", "volume_ratio"],
    )
)
panel[["trade_date", "ts_code", factor_col, "overnight_return"]].head()

In [ ]:
result = run_single_factor_analysis(panel, factor_col=factor_col, target_col="overnight_return")
ic_summary = pd.DataFrame([
    {
        "factor": factor_col,
        "mean_ic": result["ic"]["mean_ic"],
        "rank_ic": result["ic"]["rank_ic"],
        "ic_ir": result["ic"]["ic_ir"],
        "positive_rate": result["ic"]["positive_rate"],
        "coverage": result["ic"].get("coverage", 0.0),
        "rolling_1y_valid_ratio": result["ic"].get("rolling_1y_valid_ratio", 0.0),
        "long_short_mean": result["layered"]["long_short_returns"].mean() if not result["layered"]["long_short_returns"].empty else 0.0,
        "monotonicity": result["layered"].get("monotonicity", 0.0),
    }
])
ic_summary

In [ ]:
ic_series = result["ic"]["ic_series"]
group_returns = result["layered"]["group_returns"]
ic_summary.to_csv(OUTPUT_ROOT / f"02_{factor_col}_summary.csv", index=False)
ic_series.to_csv(OUTPUT_ROOT / f"02_{factor_col}_ic_series.csv", index=False)
group_returns.to_csv(OUTPUT_ROOT / f"02_{factor_col}_group_returns.csv")
ic_series.tail(), group_returns.tail()

## Interpretation Checklist

- Prefer factors with stable sign, acceptable coverage, and slow IC decay.
- If `coverage` is low, inspect whether the factor is event-driven and should be evaluated with coverage-aware thresholds.
- If group returns are not monotonic, the factor may need winsorization, neutralization, or interaction terms.